# Pipeline de Extração Oracle → ClickHouse Bronze

Este notebook executa a extração das tabelas específicas do Oracle para a camada Bronze no ClickHouse.

In [116]:
import sys
import os

is_docker = os.path.exists('/app')
if is_docker:
    project_root = '/app'
    sys.path.insert(0, '/app')
    print("✓ Ambiente Docker detectado")
else:
    current_dir = os.getcwd()
    
    possible_roots = [
        current_dir,
        os.path.dirname(current_dir),
        os.path.join(os.path.dirname(current_dir), 'track-data-platform') if 'track-data-platform' in os.path.dirname(current_dir) else None,
    ]
    
    project_root = None
    for root in possible_roots:
        if root and os.path.exists(os.path.join(root, 'connectors')):
            project_root = root
            break
    
    if not project_root:
        project_root = current_dir
    
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    
    print(f"✓ Ambiente local detectado: {project_root}")
    connectors_path = os.path.join(project_root, 'connectors')
    if os.path.exists(connectors_path):
        print(f"✓ Módulo connectors encontrado")
    else:
        print(f"⚠ Módulo connectors não encontrado em: {connectors_path}")

try:
    from datetime import datetime
    import importlib
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, concat_ws, lit, sha2, struct, to_json, current_timestamp
    from connectors import clickhouse_client
    importlib.reload(clickhouse_client)
    from connectors.clickhouse_client import ClickHouseClient
    try:
        from config.settings import oracle_config, clickhouse_config, spark_config
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv(os.path.join(project_root, '.env'))
        
        class OracleConfig:
            def __init__(self):
                self.host = os.getenv('ORACLE_HOST', 'localhost')
                self.port = int(os.getenv('ORACLE_PORT', '1521'))
                self.service = os.getenv('ORACLE_SERVICE', '')
                self.user = os.getenv('ORACLE_USER', '')
                self.password = os.getenv('ORACLE_PASSWORD', '')
        
        class ClickHouseConfig:
            def __init__(self):
                self.host = os.getenv('CLICKHOUSE_HOST', '')
                self.port = int(os.getenv('CLICKHOUSE_PORT', '8443'))
                self.user = os.getenv('CLICKHOUSE_USER', 'default')
                self.password = os.getenv('CLICKHOUSE_PASSWORD', '')
                self.database = os.getenv('CLICKHOUSE_DATABASE', 'default')
                self.secure = os.getenv('CLICKHOUSE_SECURE', 'true').lower() == 'true'
                self.verify = os.getenv('CLICKHOUSE_VERIFY', 'true').lower() == 'true'
        
        class SparkConfig:
            def __init__(self):
                self.driver_memory = os.getenv('SPARK_DRIVER_MEMORY', '4g')
                self.executor_memory = os.getenv('SPARK_EXECUTOR_MEMORY', '8g')
                self.executor_cores = int(os.getenv('SPARK_EXECUTOR_CORES', '4'))
                self.sql_shuffle_partitions = int(os.getenv('SPARK_SQL_SHUFFLE_PARTITIONS', '200'))
                self.sql_adaptive_enabled = os.getenv('SPARK_SQL_ADAPTIVE_ENABLED', 'true').lower() == 'true'
        
        oracle_config = OracleConfig()
        clickhouse_config = ClickHouseConfig()
        spark_config = SparkConfig()
    
    print("✓ Todas as dependências importadas com sucesso")
except ImportError as e:
    error_msg = str(e)
    if "pyspark" in error_msg.lower():
        if not is_docker:
            pyenv_site_packages = os.path.expanduser("~/.pyenv/versions/3.11.14/lib/python3.11/site-packages")
            if os.path.exists(pyenv_site_packages):
                if pyenv_site_packages not in sys.path:
                    sys.path.insert(0, pyenv_site_packages)
                pyspark_path = os.path.join(pyenv_site_packages, "pyspark")
                if os.path.exists(pyspark_path):
                    try:
                        from datetime import datetime
                        import importlib
                        from pyspark.sql import SparkSession
                        from pyspark.sql.functions import col, concat_ws, lit, sha2, struct, to_json, current_timestamp
                        from connectors import clickhouse_client
                        importlib.reload(clickhouse_client)
                        from connectors.clickhouse_client import ClickHouseClient
                        try:
                            from config.settings import oracle_config, clickhouse_config, spark_config
                        except ImportError:
                            from dotenv import load_dotenv
                            load_dotenv(os.path.join(project_root, '.env'))
                            
                            class OracleConfig:
                                def __init__(self):
                                    self.host = os.getenv('ORACLE_HOST', 'localhost')
                                    self.port = int(os.getenv('ORACLE_PORT', '1521'))
                                    self.service = os.getenv('ORACLE_SERVICE', '')
                                    self.user = os.getenv('ORACLE_USER', '')
                                    self.password = os.getenv('ORACLE_PASSWORD', '')
                            
                            class ClickHouseConfig:
                                def __init__(self):
                                    self.host = os.getenv('CLICKHOUSE_HOST', '')
                                    self.port = int(os.getenv('CLICKHOUSE_PORT', '8443'))
                                    self.user = os.getenv('CLICKHOUSE_USER', 'default')
                                    self.password = os.getenv('CLICKHOUSE_PASSWORD', '')
                                    self.database = os.getenv('CLICKHOUSE_DATABASE', 'default')
                                    self.secure = os.getenv('CLICKHOUSE_SECURE', 'true').lower() == 'true'
                                    self.verify = os.getenv('CLICKHOUSE_VERIFY', 'true').lower() == 'true'
                            
                            class SparkConfig:
                                def __init__(self):
                                    self.driver_memory = os.getenv('SPARK_DRIVER_MEMORY', '4g')
                                    self.executor_memory = os.getenv('SPARK_EXECUTOR_MEMORY', '8g')
                                    self.executor_cores = int(os.getenv('SPARK_EXECUTOR_CORES', '4'))
                                    self.sql_shuffle_partitions = int(os.getenv('SPARK_SQL_SHUFFLE_PARTITIONS', '200'))
                                    self.sql_adaptive_enabled = os.getenv('SPARK_SQL_ADAPTIVE_ENABLED', 'true').lower() == 'true'
                            
                            oracle_config = OracleConfig()
                            clickhouse_config = ClickHouseConfig()
                            spark_config = SparkConfig()
                        
                        print("✓ PySpark encontrado no pyenv global")
                        print("✓ Todas as dependências importadas com sucesso")
                    except ImportError as e2:
                        if "_with_origin" in str(e2) or "pyspark.errors.utils" in str(e2):
                            print("⚠ PySpark no pyenv mas incompatível com Python 3.11")
                            print("\n💡 SOLUÇÃO RECOMENDADA:")
                            print("   Execute no Jupyter Lab (PySpark configurado corretamente):")
                            print("   http://localhost:8888/lab")
                            raise
                        else:
                            print(f"✗ Erro ao importar: {e2}")
                            raise
                    except Exception as e2:
                        print(f"✗ Erro ao importar: {e2}")
                        raise
                else:
                    print(f"✗ PySpark não encontrado em: {pyspark_path}")
            else:
                print(f"✗ Caminho pyenv não encontrado: {pyenv_site_packages}")
        
        if "pyspark" in error_msg.lower():
            print(f"\n✗ Erro ao importar: {error_msg}")
            print("\n⚠ ATENÇÃO: PySpark não está instalado no ambiente atual.")
            print("\nOpções:")
            print("1. Execute no Docker (recomendado):")
            print("   docker compose up -d jupyter")
            print("   Acesse: http://localhost:8888/lab")
            print("\n2. Ou instale localmente (pode ter problemas com Python 3.11):")
            print("   pip install pyspark==3.5.0")
            raise
    else:
        print(f"✗ Erro ao importar: {error_msg}")
        raise

ref_date = datetime.now().strftime('%Y-%m-%d')
print(f"\nData de referência: {ref_date}")

✓ Ambiente Docker detectado
✓ Todas as dependências importadas com sucesso

Data de referência: 2026-01-28


In [117]:
import os
import zipfile

try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
except:
    pass

is_docker = os.path.exists('/app')

if not is_docker:
    pyspark_jars_path = os.path.expanduser('~/.pyenv/versions/3.11.14/lib/python3.11/site-packages/pyspark/jars')
    hive_jar_path = os.path.join(pyspark_jars_path, 'spark-hive_2.13-4.1.1.jar')
    
    if not os.path.exists(hive_jar_path):
        os.makedirs(pyspark_jars_path, exist_ok=True)
        with zipfile.ZipFile(hive_jar_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            zf.writestr('META-INF/MANIFEST.MF', 'Manifest-Version: 1.0\n')

spark_builder = SparkSession.builder.appName(f'OracleToBronze_{ref_date}')

if is_docker:
    spark_home = os.environ.get('SPARK_HOME', '/usr/local/spark')
    ojdbc_jar = f"{spark_home}/jars/ojdbc8-21.9.0.0.jar"
    clickhouse_jar = f"{spark_home}/jars/clickhouse-jdbc-0.4.6-all.jar"
    spark_builder = spark_builder.config('spark.jars', f"{ojdbc_jar},{clickhouse_jar}")
    print(f"✓ Usando JARs locais do Docker")
else:
    spark_builder = spark_builder.config(
        "spark.jars.packages",
        "com.oracle.database.jdbc:ojdbc8:23.2.0.0"
    )
    print("✓ Usando driver Oracle JDBC via Maven (download automático)")
    print("  (ClickHouse usa clickhouse-connect via Python, não precisa de driver JDBC)")

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_SQL_HIVE_METASTORE_JARS'] = ''
os.environ['SPARK_SQL_HIVE_METASTORE_VERSION'] = ''

spark = (
    spark_builder
    .config('spark.sql.catalogImplementation', 'in-memory')
    .config('spark.sql.warehouse.dir', '/tmp/spark-warehouse')
    .config('spark.sql.hive.metastore.enabled', 'false')
    .config('spark.sql.hive.metastore.version', '')
    .config('spark.sql.hive.metastore.jars', '')
    .config('spark.sql.hive.metastore.sharedPrefixes', '')
    .config('spark.sql.hive.metastore.barrierPrefixes', '')
    .config('spark.sql.hive.metastore.jars.path', '')
    .config('spark.sql.hive.metastore.schema.verification', 'false')
    .config('spark.sql.hive.metastore.schema.verification.record.version', 'false')
    .config('spark.driver.memory', spark_config.driver_memory)
    .config('spark.executor.memory', spark_config.executor_memory)
    .config('spark.executor.cores', spark_config.executor_cores)
    .config('spark.sql.shuffle.partitions', spark_config.sql_shuffle_partitions)
    .config('spark.sql.adaptive.enabled', str(spark_config.sql_adaptive_enabled).lower())
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.sources.provider', 'parquet')
    .getOrCreate()
)

print("✓ Spark Session criada")

✓ Spark Session criada


In [118]:
import os
from dotenv import load_dotenv

env_path = os.path.join(project_root, '.env')
load_dotenv(env_path, override=True)

ch_host = os.getenv('CLICKHOUSE_HOST', '')
ch_port = int(os.getenv('CLICKHOUSE_PORT', '8443'))
ch_user = os.getenv('CLICKHOUSE_USER', 'default')
ch_password = os.getenv('CLICKHOUSE_PASSWORD', '')
ch_database = os.getenv('CLICKHOUSE_DATABASE', 'default')
ch_secure = os.getenv('CLICKHOUSE_SECURE', 'true').lower() == 'true'
ch_verify = os.getenv('CLICKHOUSE_VERIFY', 'true').lower() == 'true'

print(f"Conectando ao ClickHouse: {ch_host}:{ch_port}")
print(f"Usuário: {ch_user}")
print(f"Database: {ch_database}")

try:
    client = ClickHouseClient(
        host=ch_host,
        port=ch_port,
        username=ch_user,
        password=ch_password,
        database=ch_database,
        secure=ch_secure,
        verify=ch_verify
    )
    
    ddl = """
    CREATE DATABASE IF NOT EXISTS bronze
    """
    client.execute_query(ddl)
    print("✓ Database bronze verificado")
except Exception as e:
    error_msg = str(e)
    if "401" in error_msg or "Authentication failed" in error_msg or "password is incorrect" in error_msg:
        print("✗ Erro de autenticação no ClickHouse")
        print("\nPossíveis causas:")
        print("1. Senha incorreta no arquivo .env")
        print("2. Usuário não existe no ClickHouse Cloud")
        print("3. Credenciais expiradas ou alteradas")
        print("\nSoluções:")
        print("1. Verifique as credenciais no ClickHouse Cloud:")
        print("   https://clickhouse.cloud/")
        print("2. Atualize o arquivo .env com as credenciais corretas")
        print("3. Ou use o Jupyter Lab onde as credenciais podem estar diferentes")
        raise
    else:
        print(f"✗ Erro ao conectar ao ClickHouse: {error_msg[:200]}")
        raise

ddl = """
CREATE TABLE IF NOT EXISTS bronze.snapshot_raw
(
    ref_date Date,
    table_name String,
    primary_key String,
    row_hash String,
    data String,
    ingestion_timestamp DateTime DEFAULT now()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name, primary_key)
SETTINGS index_granularity = 8192
"""
client.execute_query(ddl)
print("✓ Tabela bronze.snapshot_raw verificada")

client.close()

✓ Database bronze verificado
✓ Tabela bronze.snapshot_raw verificada


In [119]:
TABLES_TO_EXTRACT = [
    {"schema": "ginf", "table": "depara_cliente"},
    {"schema": "ginf", "table": "BASE_CEP_COMPLETA"},
    {"schema": "ginf", "table": "TST_CONTRATOS_BI"},
    {"schema": "siga", "table": "SC5030"},
    {"schema": "siga", "table": "SC6030"},
    {"schema": "ginf", "table": "TST_HISTORICO_SOLICITACOES"},
    {"schema": "ginf", "table": "TST_SOLICIT_CADASTRADAS"},
    {"schema": "siga", "table": "SD2030"},
    {"schema": "siga", "table": "SF2030"},
    {"schema": "siga", "table": "ZTX030"},
    {"schema": "ginf", "table": "TST_CONTRATOS"}
]

print(f"Total de tabelas para extrair: {len(TABLES_TO_EXTRACT)}")
for t in TABLES_TO_EXTRACT:
    print(f"  - {t['schema']}.{t['table']}")

Total de tabelas para extrair: 11
  - ginf.depara_cliente
  - ginf.BASE_CEP_COMPLETA
  - ginf.TST_CONTRATOS_BI
  - siga.SC5030
  - siga.SC6030
  - ginf.TST_HISTORICO_SOLICITACOES
  - ginf.TST_SOLICIT_CADASTRADAS
  - siga.SD2030
  - siga.SF2030
  - siga.ZTX030
  - ginf.TST_CONTRATOS


In [120]:
def get_primary_keys_from_oracle(spark, schema, table, oracle_jdbc_url):
    try:
        query = f"""
        SELECT column_name
        FROM all_cons_columns
        WHERE constraint_name = (
            SELECT constraint_name
            FROM all_constraints
            WHERE table_name = '{table}'
            AND owner = '{schema.upper()}'
            AND constraint_type = 'P'
            AND rownum = 1
        )
        ORDER BY position
        """
        
        pk_df = (
            spark.read.format("jdbc")
            .option("url", oracle_jdbc_url)
            .option("driver", "oracle.jdbc.OracleDriver")
            .option("query", query)
            .option("user", oracle_config.user)
            .option("password", oracle_config.password)
            .load()
        )
        
        primary_keys = [row.column_name for row in pk_df.collect()]
        return primary_keys if primary_keys else []
    except Exception as e:
        print(f"    ⚠ Não foi possível obter chaves primárias: {str(e)}")
        return []

def extract_table_to_bronze(spark, schema, table, ref_date, oracle_jdbc_url):
    extraction_start = datetime.now()
    
    print(f"\n{'='*80}")
    print(f"Extraindo: {schema}.{table}")
    print(f"{'='*80}")
    
    try:
        primary_keys = get_primary_keys_from_oracle(spark, schema, table, oracle_jdbc_url)
        
        source_df = (
            spark.read.format("jdbc")
            .option("url", oracle_jdbc_url)
            .option("driver", "oracle.jdbc.OracleDriver")
            .option("dbtable", f"{schema}.{table}")
            .option("user", oracle_config.user)
            .option("password", oracle_config.password)
            .option("fetchsize", "10000")
            .option("numPartitions", "10")
            .load()
        )
        
        if not primary_keys:
            print(f"    ⚠ Nenhuma chave primária encontrada. Usando primeiras colunas como chave.")
            all_columns = [c.name for c in source_df.schema]
            primary_keys = all_columns[:min(5, len(all_columns))]
        
        print(f"    ✓ Schema obtido: {len(source_df.schema.fields)} colunas")
        print(f"    ✓ Chaves primárias: {primary_keys}")
        
        row_count = source_df.count()
        print(f"    ✓ {row_count} registros lidos do Oracle")
        
        if row_count == 0:
            print(f"    ⚠ Tabela vazia, pulando...")
            return
        
        primary_key_expr = concat_ws('|||', *[col(pk) for pk in primary_keys])
        row_hash_expr = sha2(concat_ws('|||', *[col(c.name) for c in source_df.schema]), 256)
        json_expr = to_json(struct(*[col(c.name) for c in source_df.schema]))
        
        bronze_df = (
            source_df
            .select(
                lit(ref_date).cast('date').alias('ref_date'),
                lit(f"{schema}.{table}").alias('table_name'),
                primary_key_expr.alias('primary_key'),
                row_hash_expr.alias('row_hash'),
                json_expr.alias('data'),
                current_timestamp().alias('ingestion_timestamp')
            )
        )
        
        clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_config.host}:{clickhouse_config.port}/{clickhouse_config.database}"
        
        (
            bronze_df.write
            .format('jdbc')
            .option('url', clickhouse_jdbc_url)
            .option('dbtable', 'bronze.snapshot_raw')
            .option('user', clickhouse_config.user)
            .option('password', clickhouse_config.password)
            .option('driver', 'com.clickhouse.jdbc.ClickHouseDriver')
            .option('batchsize', '500000')
            .mode('append')
            .save()
        )
        
        print(f"    ✓ Dados inseridos no Bronze: {row_count} registros")
        
        extraction_end = datetime.now()
        duration = (extraction_end - extraction_start).total_seconds()
        print(f"    ✓ Extração concluída em {duration:.2f} segundos")
        
    except Exception as e:
        error_msg = str(e)[:500]
        print(f"    ✗ Erro ao processar {schema}.{table}: {error_msg}")
        raise

In [121]:
oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"

print("=" * 80)
print("INICIANDO EXTRAÇÃO DAS TABELAS")
print("=" * 80)

for table_config in TABLES_TO_EXTRACT:
    try:
        extract_table_to_bronze(
            spark=spark,
            schema=table_config["schema"],
            table=table_config["table"],
            ref_date=ref_date,
            oracle_jdbc_url=oracle_jdbc_url
        )
    except Exception as e:
        print(f"\n✗ Falha na extração de {table_config['schema']}.{table_config['table']}: {str(e)}")
        continue

print("\n" + "=" * 80)
print("EXTRAÇÃO CONCLUÍDA")
print("=" * 80)

    ⚠ Não foi possível obter chaves primárias: An error occurred while calling o6126.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=CvBepDZBS2OLQDTFjbvbdg==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createCo

In [122]:
client = ClickHouseClient(
    host=clickhouse_config.host,
    port=clickhouse_config.port,
    username=clickhouse_config.user,
    password=clickhouse_config.password,
    database=clickhouse_config.database,
    secure=clickhouse_config.secure,
    verify=clickhouse_config.verify
)

query = f"""
SELECT 
    table_name,
    count(*) as total_rows,
    min(ingestion_timestamp) as first_ingestion,
    max(ingestion_timestamp) as last_ingestion
FROM bronze.snapshot_raw
WHERE ref_date = toDate('{ref_date}')
GROUP BY table_name
ORDER BY table_name
"""

result = client.execute_query_with_result(query)

print(f"\nResultados da extração para {ref_date}:")
print("=" * 80)
for row in result.result_rows:
    print(f"{row[0]:40} | {row[1]:>15} registros | {row[3]}")

client.close()


Resultados da extração para 2026-01-28:


In [123]:
spark.stop()
print("✓ Spark Session finalizada")

✓ Spark Session finalizada
